In [1]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [2]:
ground_truth[10]

{'question': 'How do students join the office hours or live workshop sessions if the Zoom link isn’t shared with us?',
 'document': '489dd1c9d9'}

In [3]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [4]:
doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

In [5]:
q = ground_truth[10]
q

{'question': 'How do students join the office hours or live workshop sessions if the Zoom link isn’t shared with us?',
 'document': '489dd1c9d9'}

In [6]:
doc_idx[q['document']]

{'id': '489dd1c9d9',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?',
 'answer': 'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).\n\nDon’t post questions in chat as they may be missed if the room is very active.'}

In [7]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [8]:
from evaluation_utils import RAGWithUsage

assistant = RAGWithUsage(
    index=index,
    llm_client=openai_client,
    course='llm-zoomcamp',
)

In [9]:
rec = ground_truth[0]
question = rec["question"]

answer_llm = assistant.rag(question)
answer_llm

'Yes, you can still join the course if you just found out about it.\n\nFor the certificate, the important deadline is that you must submit your capstone project while the course is still accepting submissions. You also need to complete the required peer reviews, and this has to happen with a live cohort.\n\nHomework deadlines are not required for the certificate, and missing homework won’t prevent you from getting one.'

In [10]:
q['question']

'How do students join the office hours or live workshop sessions if the Zoom link isn’t shared with us?'

In [11]:
answer = assistant.rag(q['question'])

In [12]:
assistant.total_cost()

0.0017654999999999997

In [13]:
print(answer)

Students join the office hours or live workshop sessions via **YouTube Live**, not Zoom.

- The **Zoom link is only for instructors/presenters/TAs**
- The **live video link** is posted in the **announcements channel on Telegram and Slack** before the session starts
- You can also watch on the **DataTalksClub YouTube channel**
- Questions should be submitted through **Slido** (the link is pinned in chat when live), not in the YouTube chat


In [14]:
doc_id = q["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

answer_orig

'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).\n\nDon’t post questions in chat as they may be missed if the room is very active.'

In [15]:
rag_result = {
    "question": q['question'],
    "answer_llm": answer,
    "answer_orig": answer_orig,
    "document": doc_id,
}

rag_result

{'question': 'How do students join the office hours or live workshop sessions if the Zoom link isn’t shared with us?',
 'answer_llm': 'Students join the office hours or live workshop sessions via **YouTube Live**, not Zoom.\n\n- The **Zoom link is only for instructors/presenters/TAs**\n- The **live video link** is posted in the **announcements channel on Telegram and Slack** before the session starts\n- You can also watch on the **DataTalksClub YouTube channel**\n- Questions should be submitted through **Slido** (the link is pinned in chat when live), not in the YouTube chat',
 'answer_orig': 'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalks

In [16]:
def generate_rag_answer(rec):
    question = rec["question"]
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    answer_llm = assistant.rag(question)
    answer_orig = original_doc["answer"]

    result = {
        "question": question,
        "answer_llm": answer_llm,
        "answer_orig": answer_orig,
        "document": doc_id,
    }

    return result

In [17]:
record = generate_rag_answer(q)
record

{'question': 'How do students join the office hours or live workshop sessions if the Zoom link isn’t shared with us?',
 'answer_llm': 'Students join via **YouTube Live**, not Zoom.\n\n- The **Zoom link is only for instructors/presenters/TAs**.\n- For students, the **video URL is posted in the announcements channel on Telegram and Slack** before the session starts.\n- The live stream is also available on the **DataTalksClub YouTube channel**.\n- You can submit questions through **Slido**; the link is pinned in chat when the session is live.',
 'answer_orig': 'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).\n\nDon’t post questions in ch

In [18]:
assistant.total_cost()

0.0027472499999999997

In [19]:
assistant.reset_usage()

In [20]:
assistant.total_cost()

0.0

In [21]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [22]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, ground_truth, generate_rag_answer)

  0%|          | 0/590 [00:00<?, ?it/s]

In [23]:
results[:10]

[{'question': 'Can I still join the course if I just found out about it, and is there any deadline I should know about for getting a certificate?',
  'answer_llm': 'Yes, you can still join the course.\n\nIf you want a certificate, the important deadline is not the homework deadline, but the time when the course is still accepting **project submissions**. To get the certificate, you need to:\n\n- finish the **capstone project**\n- complete the required **peer reviews**\n- submit everything while a **live cohort** is still accepting submissions\n\nHomework is **not required** for the certificate.',
  'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'document': '74eb249bbf'},
 {'question': 'If I join late, am I still eligible for the certificate, or do I need to finish something by a certain time?',
  'answer_llm': 'Yes, but only if you finish with a live cohort. You can join late and work through t

In [24]:
df_results = pd.DataFrame(results)

In [25]:
df_results.head()

,question,answer_llm,answer_orig,document
0,Can I still join the course if I just found ou...,"Yes, you can still join the course.\n\nIf you ...","Yes, but if you want to receive a certificate,...",74eb249bbf
1,"If I join late, am I still eligible for the ce...","Yes, but only if you finish with a live cohort...","Yes, but if you want to receive a certificate,...",74eb249bbf
2,I missed the start of the course — can I enrol...,"Yes, you can still join now.\n\nIf you want a ...","Yes, but if you want to receive a certificate,...",74eb249bbf
3,"Is it too late to join this course, and does t...",I don’t know.,"Yes, but if you want to receive a certificate,...",74eb249bbf
4,"Can new students still join, and if so, what’s...","Yes, new students can still join.\n\nTo get th...","Yes, but if you want to receive a certificate,...",74eb249bbf


In [26]:
assistant.total_cost()


0.65405325

In [27]:
df_results.to_csv("data/rag-answers-new.csv", index=False)